<div style="text-align: center;">
  <span style="color: lightskyblue; font-weight: bold; font-size: 50px;">
    Test File for Object Color Detection (D435i / D405)
  </span>
</div>

<span style="color:#B8860B; font-weight:bold; font-size:50px;">
Import Library
</span>

In [1]:
import rby1_sdk as rby
import numpy as np
import rby1_lib_ysw as ysw
import logging
import sys
import time
import pyrealsense2 as rs
import cv2

np.set_printoptions(suppress=True, precision=5, linewidth=300)  

# In .ipynb, show logging 
logging.basicConfig(
    level=logging.INFO,
    format="[%(levelname)s] %(message)s",
    # format="%(asctime)s [%(levelname)s] %(message)s",
    stream=sys.stdout,
    force=True,
)

<span style="color:#B8860B; font-weight:bold; font-size:50px;">
Initialization of Vision
</span>

In [2]:
d435i = ysw.RealSenseDepthCamera(rgb_on=True, depth_on=True, pixel=(640,480), fps=30, serial_number=ysw.HEAD_D435I_SERIAL)
d435i.initialize()

d405_R = ysw.RealSenseDepthCamera(rgb_on=True, depth_on=True, pixel=(640,480), fps=30, serial_number=ysw.RIGHT_D405_SERIAL)
d405_R.initialize()

d405_L = ysw.RealSenseDepthCamera(rgb_on=True, depth_on=True, pixel=(640,480), fps=30, serial_number=ysw.LEFT_D405_SERIAL)
d405_L.initialize()

<span style="color:#B8860B; font-weight:bold; font-size:50px;">
Colored Object Detection (3D BBox)
</span>

In [35]:
# ============================================================
# Select Camera Location
location = "head"       # head / wrist_R / wrist_L
# ============================================================

if location == "head":
    cam = d435i
elif location == "wrist_R":
    cam = d405_R
elif location == "wrist_L":
    cam = d405_L
    
    
# ============================================================
# ROI setting 
# Pixel Origin: (0,0) = 왼쪽 위 꼭짓점
# ============================================================
width = 640
height = 480
pixel_roi_x = None            # None이면 x 전체 사용
pixel_roi_y = None            # None이면 y 전체 사용
pixel_roi_y = [170, height]   # y는 170~480만 사용

    
# ============================================================
# Color setting
# ============================================================   
HSV_MASK_COLOR = [(np.array([30, 40, 40]), np.array([95, 255, 255]))]
    
    
# ============================================================
# Start
# ============================================================

cam.startPipeline()

try:
    while True:
        color_frame, depth_frame = cam.getRawFrames()

        if color_frame is None or depth_frame is None:
            continue

        color_image, depth_image = cam.frameToImage(color_frame, depth_frame)
        depth_colorMap = cam.convertDepthToColorMap(depth_image, alpha=0.03)

        detected_objects, color_image, mask = cam.detectColorObjectContour3D(
            color_image=color_image,
            depth_frame=depth_frame,
            mask_list=HSV_MASK_COLOR,       # Can be changed    
            minimum_area=50,                # Can be changed     # ignore object smaller than "minimum_area"[m^2] -> "area_2d" 
            maximum_depth=2.0,              # Can be changed     # ignore distant object larger than "maximum_depth"[m] 
            pixel_roi_x=pixel_roi_x,
            pixel_roi_y=pixel_roi_y,
        )

        for obj in detected_objects:
            print(
                f"[object {obj['id']}] "
                f"center_3d={np.round(obj['center_3d'], 3)}, "
                f"bbox_3d_size={np.round(obj['bbox_3d_size'], 3)}, "
                f"area_2d={np.round(obj['area_2d'], 3)}"
            )
            
        # ============================================================
        # Crop images for imshow (ROI)
        # ============================================================
        color_image_roi = ysw.crop_image_by_roi(
            color_image,
            pixel_roi_x=pixel_roi_x,
            pixel_roi_y=pixel_roi_y
        )

        depth_colorMap_roi = ysw.crop_image_by_roi(
            depth_colorMap,
            pixel_roi_x=pixel_roi_x,
            pixel_roi_y=pixel_roi_y
        )

        mask_roi = ysw.crop_image_by_roi(
            mask,
            pixel_roi_x=pixel_roi_x,
            pixel_roi_y=pixel_roi_y
        )

        cam.imshow("color_image", color_image_roi)
        cam.imshow("depth_image", depth_colorMap_roi)
        cam.imshow("mask", mask_roi)

        key = cam.waitKey(1)
        if key == 27:
            break

finally:
    cam.stopPipeline()
    cam.destroyAllWindows()

[INFO] [Head D435i] Pipeline started.
[object 0] center_3d=[-0.101  0.254  0.671], bbox_3d_size=[0.018 0.007 0.012], area_2d=60.0
[object 1] center_3d=[-0.132  0.207  0.658], bbox_3d_size=[0.011 0.011 0.011], area_2d=67.5
[object 2] center_3d=[-0.291  0.127  0.714], bbox_3d_size=[0.062 0.05  0.11 ], area_2d=1055.0
[object 3] center_3d=[-0.158  0.137  0.723], bbox_3d_size=[0.131 0.119 0.114], area_2d=5686.0
[object 4] center_3d=[-0.101  0.076  0.753], bbox_3d_size=[0.017 0.012 0.034], area_2d=103.0
[object 5] center_3d=[-0.338  0.043  0.681], bbox_3d_size=[0.012 0.023 0.024], area_2d=148.0
[object 6] center_3d=[-0.34   0.009  0.714], bbox_3d_size=[0.007 0.008 0.011], area_2d=53.0
[object 7] center_3d=[0.068 0.094 0.75 ], bbox_3d_size=[2.763 0.724 3.069], area_2d=143317.5
[object 0] center_3d=[-0.315  0.237  0.67 ], bbox_3d_size=[0.027 0.016 0.021], area_2d=227.0
[object 1] center_3d=[-0.281  0.199  0.665], bbox_3d_size=[0.015 0.009 0.009], area_2d=59.0
[object 2] center_3d=[-0.256  0.2 